# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2: Data Collection via Web Scraping
**Author: Gautam825406**

In [ ]:
!pip install beautifulsoup4 requests pandas --quiet

In [ ]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
import numpy as np

print('All libraries imported successfully')

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out=''
    for x in table_cells.strings:
        out += x.strip()+' '
        if out.strip() == 'F9 v1.0' or out.strip() == 'F9 v1.1' or out.strip() == 'F9 FT' or out.strip() == 'F9 B4' or out.strip() == 'F9 B5':
            break
    return out.strip()

def landing_status(table_cells):
    out=[i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass=unicodedata.normalize('NFKD',table_cells.text).strip()
    if mass:
        mass.find('kg')
        new_mass=mass[0:mass.find('kg')+2]
    else:
        new_mass=0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name=' '.join(row.contents)
    if not(colunm_name.strip().isdigit()):
        colunm_name=colunm_name.strip()
        return colunm_name

print('Helper functions defined')

## Task 1: Request Falcon 9 Wikipedia page

In [ ]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
response = requests.get(static_url)
print('Status:', response.status_code)

## Task 2: Extract column names from HTML table headers

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
print('Page title:', soup.title.string)

In [ ]:
# Find all tables
html_tables = soup.find_all('table')
print('Number of tables:', len(html_tables))

In [ ]:
# Use the 3rd table (index 2)
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print('Column names extracted:')
print(column_names)

## Task 3: Create dictionary and populate with data

In [ ]:
launch_dict = dict.fromkeys(column_names)
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', 'wikitable plainrowheaders collapsible')):
    for rows in table.find_all('tr'):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False

        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            time = datatimelist[1]
            launch_dict['Time'].append(time)
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            launch_site = row[2].a.string
            launch_dict['Launch site'].append(launch_site)
            payload = row[3].a.string
            launch_dict['Payload'].append(payload)
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            orbit = row[5].a.string
            launch_dict['Orbit'].append(orbit)
            if row[6].a:
                customer = row[6].a.string
            else:
                customer = row[6].string
            launch_dict['Customer'].append(customer)
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)
            launch_dict['Flight No.'].append(flight_number)

print('Rows extracted:', extracted_row)

## Task 4: Build and save DataFrame

In [ ]:
df = pd.DataFrame({'Flight No.': launch_dict['Flight No.'],
                   'Launch site': launch_dict['Launch site'],
                   'Payload': launch_dict['Payload'],
                   'Payload mass': launch_dict['Payload mass'],
                   'Orbit': launch_dict['Orbit'],
                   'Customer': launch_dict['Customer'],
                   'Launch outcome': launch_dict['Launch outcome'],
                   'Version Booster': launch_dict['Version Booster'],
                   'Booster landing': launch_dict['Booster landing'],
                   'Date': launch_dict['Date'],
                   'Time': launch_dict['Time']})
df.head(10)

In [ ]:
print('Shape:', df.shape)
df.dtypes

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)
print('Saved spacex_web_scraped.csv')